# TYK2 Molecular Docking Pipeline

**Target:** TYK2 kinase (PDB: 6NZP)

This notebook reproduces the complete molecular docking workflow:
1. Environment setup & tool installation
2. Protein structure preparation (fetch, clean, add hydrogens)
3. Ligand preparation
4. Molecular docking (AutoDock Vina & smina)
5. Docking results analysis & visualization
6. Protein-ligand interaction fingerprints (ProLIF)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fourmodern/ddtut1/blob/main/TYK2_Molecular_Docking_Colab.ipynb)

---
## 1. Environment Setup

Install all required packages and download binary tools for Colab.

In [ ]:
#@title 1-1. Install Python Packages {display-mode: "form"}
import subprocess, sys

# Core cheminformatics
!pip install -q rdkit
!pip install -q meeko vina
!pip install -q openbabel-wheel

# Protein preparation
!pip install -q pdbfixer openmm

# Molecular visualization
!pip install -q pymol-open-source
!pip install -q py3Dmol

# Analysis
!pip install -q MDAnalysis
!pip install -q prolif

# Data & plotting
!pip install -q seaborn pandas matplotlib

print('\n=== Python packages installed ===')

In [ ]:
#@title 1-2. Download & Install Binary Tools (smina, ADFRsuite, fpocket) {display-mode: "form"}
import os, stat, urllib.request, zipfile, tarfile, shutil

BIN_DIR = '/opt/bin' if os.path.exists('/opt/bin/smina') else '/content/bin'
os.makedirs(BIN_DIR, exist_ok=True)

def make_executable(path):
    st = os.stat(path)
    os.chmod(path, st.st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

# --- smina (Vina variant with scoring) ---
SMINA_URL = 'https://sourceforge.net/projects/smina/files/smina.static/download'
smina_path = os.path.join(BIN_DIR, 'smina')
if not os.path.exists(smina_path):
    print('Downloading smina...')
    urllib.request.urlretrieve(SMINA_URL, smina_path)
    make_executable(smina_path)
    print('smina installed.')

# --- ADFRsuite (prepare_receptor, prepare_ligand) ---
ADFR_URL = 'https://ccsb.scripps.edu/adfr/download/1038/'
adfr_tar = '/tmp/ADFRsuite.tar.gz'
ADFR_DIR = os.path.join(BIN_DIR, 'ADFRsuite')
if not os.path.exists(ADFR_DIR):
    print('Downloading ADFRsuite...')
    urllib.request.urlretrieve(ADFR_URL, adfr_tar)
    with tarfile.open(adfr_tar) as tar:
        tar.extractall('/tmp/')
    # Find the extracted directory
    adfr_extracted = [d for d in os.listdir('/tmp/') if d.startswith('ADFRsuite')][0]
    try:
        os.chdir(f'/tmp/{adfr_extracted}')
        !echo 'Y' | ./install.sh -d {ADFR_DIR} -c 0
        print('ADFRsuite installed.')
    except Exception as e:
        print(f'ADFRsuite install warning: {e}')
    finally:
        os.chdir('/content')

# Create wrapper scripts for prepare_receptor and prepare_ligand
for tool in ['prepare_receptor', 'prepare_ligand']:
    wrapper = os.path.join(BIN_DIR, tool)
    adfr_tool = os.path.join(ADFR_DIR, 'bin', tool)
    if os.path.exists(adfr_tool) and not os.path.exists(wrapper):
        with open(wrapper, 'w') as f:
            f.write(f'#!/bin/bash\n{adfr_tool} "$@"\n')
        make_executable(wrapper)

# --- fpocket (binding site detection) ---
FPOCKET_PATH = shutil.which('fpocket')
if not FPOCKET_PATH:
    print('Installing fpocket...')
    !apt-get install -qq -y fpocket 2>/dev/null || \
        (cd /tmp && git clone https://github.com/Discngine/fpocket.git && \
         cd fpocket && make && cp bin/fpocket bin/dpocket bin/tpocket {BIN_DIR}/)
    print('fpocket installed.')

# Add bin to PATH
os.environ['PATH'] = BIN_DIR + ':' + os.environ['PATH']

# Verify installations
print('\n=== Verification ===')
!{BIN_DIR}/smina --version 2>&1 | head -1
print(f'prepare_receptor: {os.path.exists(os.path.join(BIN_DIR, "prepare_receptor"))}')
print(f'prepare_ligand: {os.path.exists(os.path.join(BIN_DIR, "prepare_ligand"))}')
print('\n=== Binary tools ready ===')

In [ ]:
#@title 1-3. Import Libraries {display-mode: "form"}
from pymol import cmd
import py3Dmol
from vina import Vina
from openbabel import pybel
from rdkit import Chem
from rdkit.Chem import AllChem, Draw, PandasTools
from meeko import MoleculePreparation
import MDAnalysis as mda
from MDAnalysis.coordinates import PDB
import prolif as plf
from prolif.plotting.network import LigNetwork

from pdbfixer import PDBFixer
from openmm.app import PDBFile

import os, sys, random, warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# Working directory
WORK_DIR = '/content/docking'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

BIN_DIR = '/opt/bin' if os.path.exists('/opt/bin/smina') else '/content/bin'

print('All libraries imported successfully.')

---
## 1.5 TYK2 Structure Selection

Query RCSB PDB for all available TYK2 crystal structures and select the best one based on resolution, ligand, and experimental method.

In [ ]:
#@title 1.5-1. Query PDB for TYK2 Structures {display-mode: "form"}
import requests

# RCSB PDB Search API: find all TYK2 structures
search_url = "https://search.rcsb.org/rcsbsearch/v2/query"
query = {
    "query": {
        "type": "group",
        "logical_operator": "and",
        "nodes": [
            {"type": "terminal", "service": "full_text",
             "parameters": {"value": "TYK2 tyrosine kinase"}},
            {"type": "terminal", "service": "text",
             "parameters": {"attribute": "exptl.method", "operator": "exact_match",
                            "value": "X-RAY DIFFRACTION"}}
        ]
    },
    "return_type": "entry",
    "request_options": {"results_content_type": ["experimental"], "return_all_hits": True}
}

resp = requests.post(search_url, json=query)
pdb_ids = [hit["identifier"] for hit in resp.json().get("result_set", [])]
print(f"Found {len(pdb_ids)} TYK2-related PDB entries")

# Fetch metadata for each
struct_data = []
for pdb_id in pdb_ids:
    try:
        info = requests.get(
            f"https://data.rcsb.org/rest/v1/core/entry/{pdb_id}").json()
        poly = requests.get(
            f"https://data.rcsb.org/rest/v1/core/polymer_entity/{pdb_id}/1").json()
        
        # Get ligands
        ligs_resp = requests.get(
            f"https://data.rcsb.org/rest/v1/core/nonpolymer_entity/{pdb_id}/1")
        lig_id = ""
        if ligs_resp.status_code == 200:
            lig_data = ligs_resp.json()
            lig_id = lig_data.get("pdbx_entity_nonpoly", {}).get("comp_id", "")
        
        gene = ""
        for ref in poly.get("rcsb_polymer_entity_container_identifiers", {}).get(
                "reference_sequence_identifiers", []):
            if ref.get("database_name") == "UniProt":
                gene = ref.get("database_accession", "")
                break
        
        # Filter: only keep actual TYK2 entries
        title = info.get("struct", {}).get("title", "").upper()
        org_name = poly.get("rcsb_entity_source_organism", [{}])[0].get(
            "ncbi_scientific_name", "") if poly.get("rcsb_entity_source_organism") else ""
        
        if "TYK2" in title or "P29597" in gene:
            resolution = info.get("rcsb_entry_info", {}).get("resolution_combined", [None])
            resolution = resolution[0] if resolution else None
            struct_data.append({
                "PDB": pdb_id,
                "Resolution (Å)": resolution,
                "Title": info.get("struct", {}).get("title", "")[:80],
                "Ligand": lig_id,
                "Year": info.get("rcsb_accession_info", {}).get(
                    "deposit_date", "")[:4],
                "Organism": org_name[:20],
            })
    except Exception:
        pass

tyk2_structures = pd.DataFrame(struct_data)
if not tyk2_structures.empty:
    tyk2_structures = tyk2_structures.sort_values("Resolution (Å)")
    tyk2_structures.index = range(1, len(tyk2_structures)+1)

print(f"\nConfirmed TYK2 structures: {len(tyk2_structures)}")
tyk2_structures


In [ ]:
#@title 1.5-2. Select Structure for Docking {display-mode: "form"}

SELECTED_PDB = "6NZP"  #@param {type:"string"}

# Validate selection
if not tyk2_structures.empty:
    match = tyk2_structures[tyk2_structures["PDB"] == SELECTED_PDB.upper()]
    if not match.empty:
        row = match.iloc[0]
        print(f"Selected: {row['PDB']}")
        print(f"  Resolution: {row['Resolution (Å)']} Å")
        print(f"  Ligand: {row['Ligand']}")
        print(f"  Title: {row['Title']}")
        PDB_CODE = SELECTED_PDB.lower()
    else:
        print(f"WARNING: {SELECTED_PDB} not found in TYK2 structures.")
        print(f"Using anyway — may not be a TYK2 structure.")
        PDB_CODE = SELECTED_PDB.lower()
else:
    print("PDB query failed. Using default.")
    PDB_CODE = SELECTED_PDB.lower()

print(f"\n→ PDB_CODE = \"{PDB_CODE}\" will be used for docking.")


In [ ]:
#@title 1.5-3. Resolution & Year Comparison {display-mode: "form"}

if not tyk2_structures.empty and len(tyk2_structures) > 1:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Resolution barplot
    df_plot = tyk2_structures.head(20).copy()
    colors = ["#E74C3C" if pdb == PDB_CODE.upper() else "#3498DB"
              for pdb in df_plot["PDB"]]
    axes[0].barh(df_plot["PDB"], df_plot["Resolution (Å)"],
                 color=colors, edgecolor="black", linewidth=0.3)
    axes[0].set_xlabel("Resolution (Å)")
    axes[0].set_title("TYK2 Structures by Resolution (top 20)")
    axes[0].axvline(x=2.5, color="red", linestyle="--", alpha=0.5, label="2.5 Å")
    axes[0].legend()
    axes[0].invert_yaxis()
    
    # Year distribution
    year_counts = tyk2_structures["Year"].value_counts().sort_index()
    axes[1].bar(year_counts.index, year_counts.values, color="#2ECC71",
               edgecolor="black", linewidth=0.3)
    axes[1].set_xlabel("Year"); axes[1].set_ylabel("# Structures")
    axes[1].set_title("TYK2 Structures by Deposition Year")
    plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45, ha="right")
    
    plt.tight_layout(); plt.show()
    
    # Recommendation
    best = tyk2_structures.iloc[0]
    print(f"\nRecommendation: {best['PDB']} (best resolution: {best['Resolution (Å)']} Å)")
    if best["PDB"] != PDB_CODE.upper():
        print(f"  Current selection: {PDB_CODE.upper()} — consider switching if resolution matters.")
else:
    print("Not enough structures to compare.")


---
## 2. Utility Functions

Helper functions for protein fixing, docking box calculation, and format conversion.

In [ ]:
#@title 2-1. Utility Functions {display-mode: "form"}

def getbox(selection='sele', extending=6.0, software='vina'):
    """Calculate docking box from PyMOL selection."""
    ([minX, minY, minZ], [maxX, maxY, maxZ]) = cmd.get_extent(selection)
    minX -= float(extending)
    minY -= float(extending)
    minZ -= float(extending)
    maxX += float(extending)
    maxY += float(extending)
    maxZ += float(extending)
    SizeX = maxX - minX
    SizeY = maxY - minY
    SizeZ = maxZ - minZ
    CenterX = (maxX + minX) / 2
    CenterY = (maxY + minY) / 2
    CenterZ = (maxZ + minZ) / 2
    cmd.delete('all')
    if software == 'vina':
        return {'center_x': CenterX, 'center_y': CenterY, 'center_z': CenterZ}, \
               {'size_x': SizeX, 'size_y': SizeY, 'size_z': SizeZ}
    elif software == 'ledock':
        return {'minX': minX, 'maxX': maxX}, {'minY': minY, 'maxY': maxY}, {'minZ': minZ, 'maxZ': maxZ}
    elif software == 'both':
        return ({'center_x': CenterX, 'center_y': CenterY, 'center_z': CenterZ},
                {'size_x': SizeX, 'size_y': SizeY, 'size_z': SizeZ}), \
               ({'minX': minX, 'maxX': maxX}, {'minY': minY, 'maxY': maxY}, {'minZ': minZ, 'maxZ': maxZ})


def fix_protein(filename='', addHs_pH=7.4, output='', try_renumberResidues=False):
    """Fix missing residues/atoms and add hydrogens using PDBFixer."""
    fix = PDBFixer(filename=filename)
    fix.findMissingResidues()
    fix.findNonstandardResidues()
    fix.replaceNonstandardResidues()
    fix.removeHeterogens(True)
    fix.findMissingAtoms()
    fix.addMissingAtoms()
    fix.addMissingHydrogens(addHs_pH)
    PDBFile.writeFile(fix.topology, fix.positions, open(output, 'w'))
    if try_renumberResidues:
        try:
            original = mda.Universe(filename)
            from_fix = mda.Universe(output)
            resNum = [res.resid for res in original.residues]
            for idx, res in enumerate(from_fix.residues):
                res.resid = resNum[idx]
            save = PDB.PDBWriter(filename=output)
            save.write(from_fix)
            save.close()
        except Exception:
            print('Not possible to renumber residues')


def pdbqt_to_sdf(pdbqt_file=None, output=None):
    """Convert PDBQT docking output to SDF format."""
    results = [m for m in pybel.readfile(filename=pdbqt_file, format='pdbqt')]
    out = pybel.Outputfile(filename=output, format='sdf', overwrite=True)
    for pose in results:
        pose.data.update({'Pose': pose.data['MODEL']})
        pose.data.update({'Score': pose.data['REMARK'].split()[2]})
        del pose.data['MODEL'], pose.data['REMARK'], pose.data['TORSDO']
        out.write(pose)
    out.close()


print('Utility functions defined.')

---
## 3. Pipeline Functions

Core docking workflow functions.

In [ ]:
#@title 3-1. Protein & Ligand Preparation Functions {display-mode: "form"}

def pdb_clean(pdb_code, chain='A', ref_code=None):
    """
    Fetch PDB, keep specified chain, optionally align to reference.
    Returns: (protein_pdb, ligand_mol2)
    """
    clean_pdb = f'{pdb_code}_clean.pdb'
    lig_mol2 = f'{pdb_code}_lig.mol2'

    cmd.fetch(code=pdb_code, type='pdb')
    cmd.remove(f'not chain {chain}')

    # Align to reference structure if provided
    if ref_code and ref_code != pdb_code:
        cmd.fetch(code=ref_code, type='pdb')
        cmd.remove(f'{ref_code} and not chain {chain}')
        cmd.cealign(ref_code, pdb_code)
        cmd.save(filename=f'{pdb_code}.pdb', format='pdb', selection=pdb_code)
        cmd.delete(ref_code)

    cmd.select(name='Prot', selection='polymer.protein')
    cmd.select(name='Lig', selection='organic')
    cmd.save(filename=clean_pdb, format='pdb', selection='Prot')
    cmd.save(filename=lig_mol2, format='mol2', selection='Lig')
    cmd.delete('all')

    print(f'Saved: {clean_pdb}, {lig_mol2}')
    return clean_pdb, lig_mol2


def prep_lig(lig_mol2):
    """Add hydrogens to ligand MOL2 file."""
    lig_H = lig_mol2.replace('.mol2', '_H.mol2')
    lmol = [m for m in pybel.readfile(filename=lig_mol2, format='mol2')][0]
    lmol.addh()
    lout = pybel.Outputfile(filename=lig_H, format='mol2', overwrite=True)
    lout.write(lmol)
    lout.close()
    print(f'Ligand with H: {lig_H}')
    return lig_H


def prep_prot(clean_pdb, lig_H_mol2, extending=7.0):
    """
    Fix protein (add missing atoms/H), calculate docking box from ligand position.
    Returns: (prot_H_pdb, center, size)
    """
    prot_H = clean_pdb.replace('_clean.pdb', '_clean_H.pdb')
    fix_protein(filename=clean_pdb, addHs_pH=7.2, try_renumberResidues=True, output=prot_H)

    cmd.load(filename=prot_H, format='pdb', object='prot')
    cmd.load(filename=lig_H_mol2, format='mol2', object='lig')
    center, size = getbox(selection='lig', extending=extending, software='vina')
    cmd.delete('all')

    print(f'Protein with H: {prot_H}')
    print(f'Docking box center: {center}')
    print(f'Docking box size:   {size}')
    return prot_H, center, size


def prep_pdbqt(prot_H_pdb, lig_H_mol2):
    """Convert protein and ligand to PDBQT format."""
    rec_qt = prot_H_pdb.replace('.pdb', '.pdbqt')
    lig_qt = lig_H_mol2.replace('.mol2', '.pdbqt')

    # Receptor: openbabel + strip ROOT/BRANCH for rigid receptor
    mol = list(pybel.readfile(format='pdb', filename=prot_H_pdb))[0]
    out = pybel.Outputfile(filename=rec_qt + '.tmp', format='pdbqt', overwrite=True)
    out.write(mol); out.close()
    skip_tags = ('ROOT','ENDROOT','BRANCH','ENDBRANCH','TORSDOF')
    skip_kw = ('torsion','active','between atoms','status')
    with open(rec_qt + '.tmp') as f: raw = f.readlines()
    with open(rec_qt, 'w') as f:
        for l in raw:
            if l.startswith(skip_tags): continue
            if l.startswith('REMARK') and any(k in l.lower() for k in skip_kw): continue
            f.write(l)
    os.remove(rec_qt + '.tmp')

    # Ligand: meeko with ORIGINAL 3D coordinates from MOL2
    from meeko import MoleculePreparation, PDBQTWriterLegacy
    rdmol = Chem.MolFromMol2File(lig_H_mol2, removeHs=False)
    if rdmol is None:
        # Fallback: convert MOL2 → SDF via openbabel, then read with RDKit
        tmpsdf = lig_H_mol2.replace('.mol2', '_tmp.sdf')
        lm = list(pybel.readfile(format='mol2', filename=lig_H_mol2))[0]
        lout = pybel.Outputfile(filename=tmpsdf, format='sdf', overwrite=True)
        lout.write(lm); lout.close()
        rdmol = Chem.SDMolSupplier(tmpsdf, removeHs=False)[0]
        os.remove(tmpsdf)
    if rdmol is not None:
        preparator = MoleculePreparation()
        for setup in preparator.prepare(rdmol):
            pdbqt_str, is_ok, _ = PDBQTWriterLegacy.write_string(setup)
            if is_ok:
                with open(lig_qt, 'w') as f: f.write(pdbqt_str)
                break

    print(f'Receptor PDBQT: {rec_qt}')
    print(f'Ligand PDBQT:   {lig_qt}')
    return rec_qt, lig_qt



In [ ]:
#@title 3-2. Docking Functions {display-mode: "form"}

def run_vina_dock(rec_qt, lig_qt, center, size, exhaustiveness=32, n_poses=30):
    """
    Run AutoDock Vina docking.
    Returns: SDF output file path
    """
    out_pdbqt = rec_qt.replace('_clean_H.pdbqt', '_lig_vina_out.pdbqt')
    out_sdf = out_pdbqt.replace('.pdbqt', '.sdf')

    v = Vina(sf_name='vina')
    v.set_receptor(rec_qt)
    v.set_ligand_from_file(lig_qt)
    v.compute_vina_maps(
        center=[center['center_x'], center['center_y'], center['center_z']],
        box_size=[size['size_x'], size['size_y'], size['size_z']]
    )

    energy = v.score()
    print(f'Score before minimization: {energy[0]:.3f} kcal/mol')
    energy_min = v.optimize()
    print(f'Score after minimization:  {energy_min[0]:.3f} kcal/mol')

    v.dock(exhaustiveness=exhaustiveness, n_poses=n_poses)
    v.write_poses(out_pdbqt, n_poses=n_poses, overwrite=True)
    pdbqt_to_sdf(pdbqt_file=out_pdbqt, output=out_sdf)

    print(f'\nVina docking complete: {out_sdf}')
    return out_sdf


def run_smina_dock(rec_qt, lig_qt, center, size, exhaustiveness=256, n_poses=30, label='native'):
    """
    Run smina docking (faster, with custom scoring).
    Returns: SDF output file path
    """
    out_sdf = rec_qt.replace('_clean_H.pdbqt', f'_{label}_smina_out.sdf')

    !{BIN_DIR}/smina -r {rec_qt} -l {lig_qt} -o {out_sdf} \
        --center_x {center['center_x']} --center_y {center['center_y']} --center_z {center['center_z']} \
        --size_x {size['size_x']} --size_y {size['size_y']} --size_z {size['size_z']} \
        --exhaustiveness {exhaustiveness} --num_modes {n_poses} > /dev/null 2>&1

    print(f'smina docking complete: {out_sdf}')
    return out_sdf


def sdf_to_df(sdf_path):
    """Load SDF docking results into a Pandas DataFrame."""
    df = PandasTools.LoadSDF(sdf_path, smilesName='SMILES', molColName='Molecule',
                             includeFingerprints=True, strictParsing=False)
    # Normalize score column name
    if 'Score' in df.columns:
        df = df.rename(columns={'Score': 'minimizedAffinity'})
    if 'minimizedAffinity' in df.columns:
        df['minimizedAffinity'] = df['minimizedAffinity'].astype('float32')
    return df


print('Docking functions defined.')

---
## 4. Run Docking Pipeline for TYK2 (6NZP)

**Target:** TYK2 JH2 pseudokinase domain (PDB: 6NZP, chain A)

In [ ]:
#@title 4-1. Fetch & Clean Protein Structure {display-mode: "form"}
# PDB_CODE is set in section 1.5-2 above (default: 6nzp)
PDB_CODE = PDB_CODE if 'PDB_CODE' in dir() else '6nzp'
CHAIN = 'A'

clean_pdb, lig_mol2 = pdb_clean(PDB_CODE, chain=CHAIN)
print(f'\nProtein: {clean_pdb}')
print(f'Co-crystallized ligand: {lig_mol2}')

In [ ]:
#@title 4-2. Prepare Ligand (add hydrogens) {display-mode: "form"}
lig_H = prep_lig(lig_mol2)

In [ ]:
#@title 4-3. Prepare Protein (fix atoms, add H, calculate docking box) {display-mode: "form"}
prot_H, center, size = prep_prot(clean_pdb, lig_H, extending=7.0)
print(f'\nCenter: {center}')
print(f'Size:   {size}')

In [ ]:
#@title 4-4. Convert to PDBQT format {display-mode: "form"}
rec_qt, lig_qt = prep_pdbqt(prot_H, lig_H)

In [ ]:
#@title 4-5. Run AutoDock Vina Docking (~5 min) {display-mode: "form"}
vina_sdf = run_vina_dock(rec_qt, lig_qt, center, size, exhaustiveness=32, n_poses=30)

In [ ]:
#@title 4-6. Run smina Docking (~1 min) {display-mode: "form"}
smina_sdf = run_smina_dock(rec_qt, lig_qt, center, size, exhaustiveness=256, n_poses=30, label='native')

---
## 5. Docking Results Analysis

In [ ]:
#@title 5-1. Load Results & Compare Scores {display-mode: "form"}
# Load Vina results
df_vina = sdf_to_df(vina_sdf)
df_vina['Method'] = 'Vina'
print(f'Vina poses: {len(df_vina)}')
print(f'Vina best score: {df_vina["minimizedAffinity"].min():.2f} kcal/mol')

# Load smina results
df_smina = sdf_to_df(smina_sdf)
df_smina['Method'] = 'smina'
print(f'\nsmina poses: {len(df_smina)}')
print(f'smina best score: {df_smina["minimizedAffinity"].min():.2f} kcal/mol')

# Combined
df_all = pd.concat([df_vina, df_smina], ignore_index=True)
df_all[['Pose', 'minimizedAffinity', 'Method']].head(10)

In [ ]:
#@title 5-2. Score Distribution Boxplot {display-mode: "form"}
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Vina scores
axes[0].barh(range(len(df_vina)), df_vina['minimizedAffinity'].values, color='steelblue')
axes[0].set_xlabel('Binding Affinity (kcal/mol)')
axes[0].set_ylabel('Pose')
axes[0].set_title('AutoDock Vina Scores')

# Comparison boxplot
sns.boxplot(data=df_all, x='Method', y='minimizedAffinity', ax=axes[1], palette='Set2')
axes[1].set_ylabel('Binding Affinity (kcal/mol)')
axes[1].set_title('Vina vs smina Score Comparison')

plt.tight_layout()
plt.show()

---
## 6. 3D Visualization

In [ ]:
#@title 6-1. View Best Docking Pose (py3Dmol) {display-mode: "form"}

def docking_view(receptor_pdb, sdf_file, pose_index=0):
    """Visualize docking pose with py3Dmol."""
    view = py3Dmol.view(width=800, height=600)
    view.removeAllModels()
    view.setViewStyle({'style': 'outline', 'color': 'black', 'width': 0.1})

    # Protein
    view.addModel(open(receptor_pdb).read(), 'pdb')
    prot = view.getModel()
    prot.setStyle({'cartoon': {'arrows': True, 'tubes': True, 'style': 'oval', 'color': 'white'}})
    view.addSurface(py3Dmol.VDW, {'opacity': 0.6, 'color': 'white'})

    # Docked ligand
    poses = [mol for mol in Chem.SDMolSupplier(sdf_file, True)]
    if pose_index < len(poses) and poses[pose_index] is not None:
        pose_block = Chem.MolToMolBlock(poses[pose_index])
        view.addModel(pose_block, 'mol')
        lig = view.getModel()
        lig.setStyle({'stick': {'colorscheme': 'greenCarbon', 'radius': 0.3}})

    view.zoomTo()
    return view

# Show best Vina pose
print('Best Vina docking pose:')
docking_view(prot_H, vina_sdf, pose_index=0)

In [ ]:
#@title 6-2. View Reference Ligand + Docked Pose {display-mode: "form"}

def compare_view(receptor_pdb, ref_mol2, sdf_file, pose_index=0):
    """Compare reference ligand (magenta) with docked pose (green)."""
    view = py3Dmol.view(width=800, height=600)
    view.removeAllModels()
    view.setViewStyle({'style': 'outline', 'color': 'black', 'width': 0.1})

    # Protein
    view.addModel(open(receptor_pdb).read(), 'pdb')
    prot = view.getModel()
    prot.setStyle({'cartoon': {'arrows': True, 'tubes': True, 'style': 'oval', 'color': 'white'}})
    view.addSurface(py3Dmol.VDW, {'opacity': 0.6, 'color': 'white'})

    # Reference ligand (magenta)
    view.addModel(open(ref_mol2).read(), 'mol2')
    ref_m = view.getModel()
    ref_m.setStyle({}, {'stick': {'colorscheme': 'magentaCarbon', 'radius': 0.2}})

    # Docked pose (green)
    poses = [mol for mol in Chem.SDMolSupplier(sdf_file, True)]
    if pose_index < len(poses) and poses[pose_index] is not None:
        pose_block = Chem.MolToMolBlock(poses[pose_index])
        view.addModel(pose_block, 'mol')
        lig = view.getModel()
        lig.setStyle({}, {'stick': {'colorscheme': 'greenCarbon', 'radius': 0.2}})

    view.zoomTo()
    return view

print('Magenta = co-crystallized ligand, Green = best docked pose')
compare_view(prot_H, lig_H, vina_sdf, pose_index=0)

---
## 7. Protein-Ligand Interaction Fingerprints (ProLIF)

In [ ]:
#@title 7-1. Generate Interaction Fingerprints {display-mode: "form"}

# Load protein
prot_mol = Chem.MolFromPDBFile(prot_H, removeHs=False, sanitize=False)
prot_plf = plf.Molecule(prot_mol)
print(f'Protein residues: {prot_plf.n_residues}')

# Load docked ligands from Vina SDF
lig_suppl = list(plf.sdf_supplier(vina_sdf))
print(f'Docked poses: {len(lig_suppl)}')

# Calculate fingerprints
fp = plf.Fingerprint()
fp.run_from_iterable(lig_suppl, prot_plf, n_jobs=1)
print(f'\nInteraction types: {fp.interactions}')

In [ ]:
#@title 7-2. Interaction DataFrame {display-mode: "form"}
results_df = fp.to_dataframe()
results_df.T

In [ ]:
#@title 7-3. Interaction Network Visualization {display-mode: "form"}
# Show interaction network for best pose
net = LigNetwork.from_fingerprint(
    results_df,
    lig_suppl[0],
    kind='aggregate',
    threshold=0.3,
    rotation=270
)
net.display()

In [ ]:
#@title 7-4. Per-frame Interaction Network {display-mode: "form"}
# Show interaction network for a specific pose
POSE_IDX = 0  #@param {type:"integer"}

net_frame = LigNetwork.from_fingerprint(
    results_df,
    lig_suppl[POSE_IDX],
    kind='frame',
    frame=POSE_IDX,
    rotation=270
)
net_frame.display()

In [ ]:
#@title 7-5. Save Interaction Network as HTML {display-mode: "form"}
output_html = f'{PDB_CODE}_interactions.html'
net.save(output_html)
print(f'Saved: {output_html}')

# Download in Colab
try:
    from google.colab import files
    files.download(output_html)
except ImportError:
    pass

---
## 7.5 Advanced Docking Analysis

RMSD validation, ligand efficiency, consensus scoring, interaction frequency.

In [ ]:
#@title 7.5-1. RMSD Validation (Docked vs Crystal Pose) {display-mode: "form"}
from rdkit.Chem import rdMolAlign

def calc_rmsd_to_ref(ref_mol2, docked_sdf, max_poses=10):
    """RMSD between crystal ligand and docked poses (heavy atoms)."""
    ref = Chem.MolFromMol2File(ref_mol2, removeHs=False)
    if ref is None:
        # Fallback: load via pybel -> PDB -> RDKit
        pmol = list(pybel.readfile(format="mol2", filename=ref_mol2))[0]
        pmol.write("pdb", "/tmp/_ref.pdb", overwrite=True)
        ref = Chem.MolFromPDBFile("/tmp/_ref.pdb", removeHs=False)
    if ref is None:
        return pd.DataFrame()
    ref_noH = Chem.RemoveHs(ref)
    rows = []
    for i, mol in enumerate(Chem.SDMolSupplier(docked_sdf, removeHs=False)):
        if mol is None or i >= max_poses: continue
        try:
            mol_noH = Chem.RemoveHs(mol)
            rmsd = rdMolAlign.GetBestRMS(ref_noH, mol_noH)
            rows.append({"pose": i, "RMSD": round(rmsd, 2)})
        except Exception:
            rows.append({"pose": i, "RMSD": np.nan})
    return pd.DataFrame(rows)

# Calculate RMSD for Vina and smina results
rmsd_vina = calc_rmsd_to_ref(lig_H, vina_sdf)
rmsd_smina = calc_rmsd_to_ref(lig_H, smina_sdf)

fig, ax = plt.subplots(figsize=(10, 4))
if not rmsd_vina.empty:
    ax.bar(rmsd_vina["pose"] - 0.2, rmsd_vina["RMSD"], 0.4, label="Vina", color="steelblue")
if not rmsd_smina.empty:
    ax.bar(rmsd_smina["pose"] + 0.2, rmsd_smina["RMSD"], 0.4, label="smina", color="coral")
ax.axhline(y=2.0, color="red", linestyle="--", label="2.0 \u00c5 threshold")
ax.set_xlabel("Pose"); ax.set_ylabel("RMSD (\u00c5)")
ax.set_title("Re-docking Validation: RMSD to Crystal Pose")
ax.legend()
plt.tight_layout(); plt.show()

print("Poses with RMSD < 2.0 \u00c5 are considered successful re-docking.")
if not rmsd_vina.empty:
    n_good = (rmsd_vina["RMSD"] < 2.0).sum()
    print(f"Vina:  {n_good}/{len(rmsd_vina)} poses below 2.0 \u00c5")
if not rmsd_smina.empty:
    n_good = (rmsd_smina["RMSD"] < 2.0).sum()
    print(f"smina: {n_good}/{len(rmsd_smina)} poses below 2.0 \u00c5")


In [ ]:
#@title 7.5-2. Ligand Efficiency (LE & BEI) {display-mode: "form"}

def add_ligand_efficiency(df):
    """Add LE, BEI, NSEI columns to docking scores DataFrame."""
    for i, row in df.iterrows():
        mol = row.get("Molecule")
        if mol is not None:
            ha = mol.GetNumHeavyAtoms()
            mw = Chem.Descriptors.MolWt(mol)
            score = row["minimizedAffinity"]
            df.loc[i, "HA"] = ha
            df.loc[i, "MW"] = round(mw, 1)
            df.loc[i, "LE"] = round(-score / ha, 3) if ha > 0 else 0
            df.loc[i, "BEI"] = round(-score / (mw/1000), 2) if mw > 0 else 0
    return df

from rdkit.Chem import Descriptors
df_vina_le = add_ligand_efficiency(df_vina.copy())
df_smina_le = add_ligand_efficiency(df_smina.copy())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, df_le, name in [(axes[0], df_vina_le, "Vina"), (axes[1], df_smina_le, "smina")]:
    if "LE" in df_le.columns:
        ax.scatter(df_le["minimizedAffinity"], df_le["LE"], s=60, alpha=0.7)
        ax.set_xlabel("Binding Affinity (kcal/mol)")
        ax.set_ylabel("Ligand Efficiency (LE)")
        ax.set_title(f"{name}: Score vs LE")
        ax.axhline(y=0.3, color="green", linestyle="--", label="LE=0.3 (good)")
        ax.legend()
plt.tight_layout(); plt.show()

print("LE > 0.3 = drug-like efficiency, LE > 0.4 = excellent")


In [ ]:
#@title 7.5-3. Consensus Scoring (Multiple Scoring Functions) {display-mode: "form"}

scoring_functions = ["vina", "vinardo", "ad4_scoring"]
consensus_results = []

for sf in scoring_functions:
    out_sdf = f"{PDB_CODE}_{sf}_rescore.sdf"
    ret = os.system(
        f'{BIN_DIR}/smina -r {rec_qt} -l {lig_qt} -o {out_sdf} '
        f'--center_x {center["center_x"]} --center_y {center["center_y"]} '
        f'--center_z {center["center_z"]} '
        f'--size_x {size["size_x"]} --size_y {size["size_y"]} --size_z {size["size_z"]} '
        f'--scoring {sf} --exhaustiveness 32 --num_modes 10 > /dev/null 2>&1')
    if os.path.exists(out_sdf) and os.path.getsize(out_sdf) > 0:
        df_sf = sdf_to_df(out_sdf)
        if not df_sf.empty and "minimizedAffinity" in df_sf.columns:
            for _, row in df_sf.iterrows():
                consensus_results.append({"SF": sf, "pose": row.get("Pose", "?"),
                                          "score": row["minimizedAffinity"]})

if consensus_results:
    cons_df = pd.DataFrame(consensus_results)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.boxplot(data=cons_df, x="SF", y="score", palette="Set2", ax=axes[0])
    axes[0].set_ylabel("Score (kcal/mol)"); axes[0].set_title("Consensus Scoring Comparison")
    # Rank correlation
    pivot = cons_df.groupby("SF")["score"].apply(list)
    min_len = min(len(v) for v in pivot.values)
    rank_df = pd.DataFrame({sf: pd.Series(scores[:min_len]).rank()
                            for sf, scores in pivot.items()})
    sns.heatmap(rank_df.corr("spearman"), annot=True, fmt=".2f", cmap="coolwarm",
               ax=axes[1], vmin=0, vmax=1)
    axes[1].set_title("Spearman Rank Correlation Between SFs")
    plt.tight_layout(); plt.show()
else:
    print("Consensus scoring failed.")


In [ ]:
#@title 7.5-4. Interaction Frequency Heatmap (All Poses) {display-mode: "form"}

# Load protein and all poses for ProLIF
prot_mol = Chem.MolFromPDBFile(prot_H, removeHs=False, sanitize=False)
if prot_mol is None:
    prot_u = mda.Universe(prot_H, guess_bonds=True)
    prot_mol = plf.Molecule.from_mda(prot_u)
else:
    prot_mol = plf.Molecule(prot_mol)

# Use Vina results (all poses)
all_ligs = list(plf.sdf_supplier(vina_sdf))
fp_all = plf.Fingerprint()
fp_all.run_from_iterable(all_ligs, prot_mol, n_jobs=1)
df_all_ifp = fp_all.to_dataframe()

# Interaction frequency: fraction of poses with each interaction
freq = df_all_ifp.mean().reset_index()
freq.columns = ["interaction", "frequency"]
freq = freq[freq["frequency"] > 0.1].sort_values("frequency", ascending=False)

# Parse residue and interaction type
freq["residue"] = freq["interaction"].apply(lambda x: str(x[1]) if isinstance(x, tuple) else str(x))
freq["type"] = freq["interaction"].apply(lambda x: str(x[2]) if isinstance(x, tuple) else "")

# Pivot for heatmap
try:
    df_bv = fp_all.to_bitvectors()
    freq_matrix = pd.DataFrame(df_bv).mean()
    # Simple bar chart of top interactions
    fig, ax = plt.subplots(figsize=(14, 5))
    top = freq.head(20)
    colors = {"Hydrophobic": "#3498DB", "HBDonor": "#E74C3C", "HBAcceptor": "#2ECC71",
              "PiStacking": "#9B59B6", "PiCation": "#F39C12", "CationPi": "#F39C12",
              "Cationic": "#E67E22", "Anionic": "#1ABC9C"}
    bar_colors = [colors.get(t, "gray") for t in top["type"]]
    ax.barh(range(len(top)), top["frequency"], color=bar_colors, edgecolor="black", linewidth=0.3)
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels([f'{r} ({t})' for r, t in zip(top["residue"], top["type"])], fontsize=9)
    ax.set_xlabel("Frequency (fraction of poses)")
    ax.set_title(f"Top Interaction Frequencies Across {len(all_ligs)} Poses")
    ax.axvline(x=0.5, color="red", linestyle="--", alpha=0.5, label="50% threshold")
    ax.legend()
    plt.tight_layout(); plt.show()
except Exception as e:
    print(f"Frequency heatmap: {e}")
    print("Showing raw interaction DataFrame instead:")
    display(df_all_ifp.T)


---
## 8. Virtual Screening with Custom Ligands (Optional)

Dock your own ligands against TYK2 using smina.

In [ ]:
#@title 8-1. Prepare Custom Ligands from SMILES {display-mode: "form"}

# Example: provide your SMILES list here
smiles_list = [
    ('Ligand_1', 'c1ccc2c(c1)c1ccccc1[nH]2'),  # carbazole (example)
    ('Ligand_2', 'c1ccc(-c2ccncc2)cc1'),         # 4-phenylpyridine (example)
]

# Or load from CSV:
# df_ligs = pd.read_csv('my_ligands.csv')  # columns: id, smiles
# smiles_list = list(zip(df_ligs['id'], df_ligs['smiles']))

mols = []
for name, smi in smiles_list:
    mol = Chem.MolFromSmiles(smi)
    if mol:
        mol = Chem.AddHs(mol)
        AllChem.EmbedMolecule(mol, randomSeed=42)
        AllChem.MMFFOptimizeMolecule(mol)
        mol.SetProp('_Name', name)
        mol.SetProp('ID', name)
        mols.append(mol)

# Write to SDF
custom_sdf = 'custom_ligands.sdf'
writer = Chem.SDWriter(custom_sdf)
for mol in mols:
    writer.write(mol)
writer.close()
print(f'Prepared {len(mols)} ligands -> {custom_sdf}')

In [ ]:
#@title 8-2. Dock Custom Ligands with smina {display-mode: "form"}
custom_out_sdf = run_smina_dock(rec_qt, custom_sdf, center, size,
                                exhaustiveness=128, n_poses=10, label='custom')

df_custom = sdf_to_df(custom_out_sdf)
print(f'\nResults: {len(df_custom)} poses')
df_custom[['minimizedAffinity']].describe()

In [ ]:
#@title 8-3. Score Boxplot by Ligand {display-mode: "form"}
if 'ID' in df_custom.columns and len(df_custom['ID'].unique()) > 1:
    fig, ax = plt.subplots(figsize=(10, 6))
    df2 = pd.DataFrame({col: vals['minimizedAffinity'] for col, vals in df_custom.groupby('ID')})
    meds = df2.min().sort_values(ascending=False)
    df2[meds.index].boxplot(ax=ax, vert=False)
    ax.set_xlabel('Binding Affinity (kcal/mol)')
    ax.set_title('Custom Ligand Docking Scores')
    plt.tight_layout()
    plt.show()
else:
    print('Only one ligand — skipping boxplot comparison.')
    print(df_custom[['minimizedAffinity']].head(10))

---
## 9. Blind Docking with fpocket (Optional)

Detect binding pockets automatically when the binding site is unknown.

In [ ]:
#@title 9-1. Detect Pockets with fpocket {display-mode: "form"}
import glob

# Run fpocket
!fpocket -f {prot_H}

# Parse results
fpocket_dir = prot_H.replace('.pdb', '_out')
pocket_files = sorted(glob.glob(f'{fpocket_dir}/pockets/*.pdb'))
print(f'\nDetected {len(pocket_files)} pockets:')
for pf in pocket_files[:5]:
    pocket_name = os.path.basename(pf)
    # Read pocket center
    cmd.load(pf, 'pocket')
    pocket_center, pocket_size = getbox(selection='pocket', extending=5.0, software='vina')
    print(f'  {pocket_name}: center={pocket_center}')

---
## 10. Download Results

In [ ]:
#@title 10-1. List Output Files {display-mode: "form"}
import glob

print('=== Generated Files ===')
for ext in ['*.pdb', '*.pdbqt', '*.mol2', '*.sdf', '*.html']:
    files = glob.glob(ext)
    if files:
        print(f'\n{ext}:')
        for f in sorted(files):
            size_mb = os.path.getsize(f) / (1024*1024)
            print(f'  {f} ({size_mb:.2f} MB)')

In [ ]:
#@title 10-2. Download Results (Colab) {display-mode: "form"}
import shutil

# Create zip of all results
output_zip = f'/content/{PDB_CODE}_docking_results'
shutil.make_archive(output_zip, 'zip', WORK_DIR)
print(f'Created: {output_zip}.zip')

try:
    from google.colab import files
    files.download(f'{output_zip}.zip')
except ImportError:
    print('Not in Colab — find results in:', WORK_DIR)